# HanziGen - 字型生成训练（本地机版）

> 本 notebook 适用于**在自己的电脑上**运行训练，无需任何云平台。
>
> 与云端版（`hanzigen_cloudstudio.ipynb` / `hanzigen_colab.ipynb` / `hanzigen_moda.ipynb`）的区别：
> - **硬件档位用 conservative（本地稳妥档）**：workers 留 2 核余量、预取保守、显存预留更稳，避免影响你日常使用、防止 OOM。
> - **内置硬件适配性检测**：Cell 1 会自动判断你的电脑是否满足训练要求（需要 NVIDIA GPU 或 Apple Silicon；AMD 核显/纯 CPU 会提示无法训练）。
> - 无需挂载网盘、无需 clone，**直接在当前项目目录运行**。

## 前置要求

1. 已安装 Python 3.10+ 与依赖：`pip install -r requirements.txt`
2. 有可用的训练加速器：**NVIDIA GPU（建议显存 >= 8GB）** 或 **Apple Silicon (MPS)**
   - 仅 AMD 核显 / 纯 CPU 的机器**无法运行训练**（本项目模型依赖 CUDA/MPS）
3. 目标字体 `.ttf` / `.otf` 已放入 `fonts/` 目录

## 流程

```
Cell 0: 配置字体名 + 阶段开关 + 补字基准 + 显存冗余量
Cell 1: 环境自检 + 硬件适配性检测 + 断连自检（可重复运行）
Cell 2: 数据准备（分析字体 → 生成数据集 → 提取字集）
Cell 3: 训练 VQ-VAE（conservative 档）
Cell 4-前置操作: 离线放置 VGG16 权重（可选，联网下载慢时运行）
Cell 4: 训练 LDM（conservative 档）
Cell 5: 推理（按 Cell 0 的补字基准补字）+ 指标 + 转 SVG
```

---
## Cell 0: 配置参数

> **只改这里！** 填你放入 `fonts/` 的字体文件名，并选择要补哪些字（补字基准）。

In [ ]:
# ==================== 修改你的字体文件名 ====================
TARGET_FONT = "myfont.ttf"    # 改成你放入 fonts/ 的字体名
# ==========================================================

FONT_NAME = TARGET_FONT.rsplit(".", 1)[0]

# ==================== 训练阶段开关（默认全开）====================
DO_DATA_PREP = True       # Cell 2: 数据准备
DO_TRAIN_VQVAE = True     # Cell 3: 训练 VQ-VAE
DO_TRAIN_LDM = True       # Cell 4: 训练 LDM
DO_INFERENCE = True       # Cell 5: 推理+指标+转SVG
# ==========================================================

# ==================== 补字基准（Cell 5 推理阶段生效）====================
# "jf7000" = jf7000 当务字集缺失字（默认，约 8,349 字基准，项目原生设计）
# "unihan" = Unihan 全字集缺失字（9 万+ 字基准，生成量大，注意耗时）
# "gbk"    = GBK 简体标准字符集缺失字（20,902 字基准，简体用户推荐）
# "gb2312" = 仅 GB2312 简体核心字（6,763 字基准，范围最保守）
CHARSET_BASE = "jf7000"
# ==========================================================

# ==================== 显存冗余量（Cell 3 训练 VQ-VAE 生效）====================
# 显存预留比例 = 允许用于训练的那部分显存占比；留的冗余 = 1 - 该比例
#   None  = 跟随档位默认（本地 conservative 档 0.85，即留 15% 冗余）
#   0.70  = 留 30% 冗余（显存较小 / 常 OOM / 训练时还要用这台电脑，推荐先试这个）
#   0.92  = 只留 8% 冗余（显存充裕且独占，追求速度）
# 取值范围 0.1-1.0。仅影响 VQ-VAE 的 batch 推算；LDM 在 latent 空间训练不受影响。
VRAM_RESERVE_FRACTION = None
# ==========================================================

STATE_FILE = "colab_state.json"

print(f"目标字体: {TARGET_FONT}")
print(f"字体名称: {FONT_NAME}")
print(f"阶段开关: 数据准备={DO_DATA_PREP} VQVAE={DO_TRAIN_VQVAE} LDM={DO_TRAIN_LDM} 推理={DO_INFERENCE}")
print(f"补字基准: {CHARSET_BASE}")
print(f"显存预留比例: {VRAM_RESERVE_FRACTION if VRAM_RESERVE_FRACTION is not None else 'auto（跟随 conservative 档 0.85）'}")

---
## Cell 1: 环境自检 + 硬件适配性检测 + 断连自检

> 会先检测你的电脑配置是否满足训练要求，再改写续训参数。可重复运行（幂等）。

In [ ]:
import os, json, re, glob
import torch
from utils.hardware import check_training_viability, detect_hardware

print("===== 硬件适配性检测 =====")
info = detect_hardware()
print(f"  CPU 核数: {info['cpu_cores']}")
if info["gpu_available"]:
    print(f"  GPU: {info['gpu_name']} ({info['vram_gb']:.1f} GB)")
elif info["mps_available"]:
    print(f"  GPU: {info['gpu_name']}（Apple Silicon MPS）")
else:
    print("  GPU: 未检测到（仅 CPU / 核显）")
print(f"  PyTorch: {torch.__version__} | CUDA: {torch.version.cuda or '无'}")

viability = check_training_viability()
if viability["viable"]:
    print(f"\n[OK] 配置可训练：{viability['reason']}")
else:
    print(f"\n[无法训练] {viability['reason']}")
    print("  → 请改用带 NVIDIA GPU 或 Apple Silicon 的机器，或使用云端 GPU 实例")
    print("    （云端请用 hanzigen_cloudstudio.ipynb / hanzigen_colab.ipynb / hanzigen_moda.ipynb）")
    print("  → 数据准备（Cell 2）与 SVG 转换（Cell 5 后半）仍可在此机器运行，但训练（Cell 3/4）无法执行。")

# ===== 断连自检 + 字体切换检测 + 自动续训改写 =====
print("\n===== 断连自检 + 字体切换检测 =====")
os.makedirs("checkpoints", exist_ok=True)

def load_state() -> dict:
    if os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}

def save_state(state: dict) -> None:
    with open(STATE_FILE, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

state = load_state()

# 字体切换检测：数据集状态与字体名绑定，防止新旧字体数据混合
prev_data_font = state.get("data_font")
if prev_data_font and prev_data_font != FONT_NAME:
    print(f"  [字体切换] {prev_data_font} → {FONT_NAME}")
    print("    · Cell 2 将重新执行数据准备（旧字体图像会被自动删除）")
    state["data_prep_done"] = False
    state["data_font"] = None

state.setdefault("font", FONT_NAME)
save_state(state)

vqvae_ckpt = f"checkpoints/vqvae_{FONT_NAME}.pth"
ldm_ckpt = f"checkpoints/ldm_{FONT_NAME}.pth"
have_vqvae = os.path.exists(vqvae_ckpt)
have_ldm = os.path.exists(ldm_ckpt)

print(f"  数据准备: {'已完成' if state.get('data_prep_done') else '未完成'}（绑定字体: {state.get('data_font') or '无'}）")
print(f"  VQ-VAE 检查点: {'存在' if have_vqvae else '不存在'}")
print(f"  LDM 检查点:    {'存在' if have_ldm else '不存在'}")

# 自动精确续训：改写 local 脚本的 RESUME_FROM
def _set_resume_from(sh_path: str, ckpt: str) -> None:
    with open(sh_path, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r'RESUME_FROM="[^"]*"', f'RESUME_FROM="{ckpt}"', content)
    with open(sh_path, "w", encoding="utf-8") as f:
        f.write(content)

_set_resume_from("scripts/train_vqvae_local.sh", vqvae_ckpt if have_vqvae else "")
_set_resume_from("scripts/train_ldm_local.sh", ldm_ckpt if have_ldm else "")
print(f"  [续训] train_vqvae_local.sh RESUME_FROM -> {vqvae_ckpt if have_vqvae else '(空，从零训练)'}")
print(f"  [续训] train_ldm_local.sh    RESUME_FROM -> {ldm_ckpt if have_ldm else '(空，从零训练)'}")

# 数据准备脚本按 CPU 核数调渲染并行度
cpu_cores = info["cpu_cores"]
render_workers = max(2, min(16, cpu_cores))
def set_sh_var(sh_path, var, value):
    with open(sh_path, encoding="utf-8") as f:
        lines = f.readlines()
    for i, ln in enumerate(lines):
        if ln.startswith(var + "="):
            body = ln[len(var)+1:].rstrip("\n")
            tail = ""
            if "#" in body:
                tail = "  " + body[body.index("#"):]
            lines[i] = f"{var}={value}{tail}\n"
            break
    with open(sh_path, "w", encoding="utf-8") as f:
        f.writelines(lines)

set_sh_var("scripts/prepare_dataset.sh", "NUM_WORKERS", render_workers)
# 本地机若无 CUDA，extract_charset 切 cpu（该步无实际张量计算）
set_sh_var("scripts/extract_charset.sh", "DEVICE", '"cuda"' if info["gpu_available"] else '"cpu"')

# 显存冗余量：写入本地训练脚本（None → auto，跟随 conservative 档）
from utils.hardware import auto_vqvae_batch_size, resolve_vram_reserve_fraction

vram_reserve_value = "auto" if VRAM_RESERVE_FRACTION is None else VRAM_RESERVE_FRACTION
set_sh_var("scripts/train_vqvae_local.sh", "VRAM_RESERVE_FRACTION", vram_reserve_value)
vram_frac = resolve_vram_reserve_fraction("conservative", VRAM_RESERVE_FRACTION)
est_batch = auto_vqvae_batch_size(info["vram_gb"], preset="conservative",
                                  vram_reserve_fraction=VRAM_RESERVE_FRACTION)
print(f"  显存预留比例: {vram_frac:.2f}（{'auto 跟随档位' if VRAM_RESERVE_FRACTION is None else 'Cell 0 指定'}）")
print(f"  显存 {info['vram_gb']:.1f} GB → 预计 VQ-VAE batch_size ≈ {est_batch}")

# 训练脚本（本地 conservative 档）
VQVAE_TRAIN_SCRIPT = "scripts/train_vqvae_local.sh"
LDM_TRAIN_SCRIPT = "scripts/train_ldm_local.sh"

print(f"\n[训练脚本] VQ-VAE → {VQVAE_TRAIN_SCRIPT}（conservative 本地档）")
print(f"[训练脚本] LDM     → {LDM_TRAIN_SCRIPT}（conservative 本地档）")
print("\n全部初始化完成！")

---
## Cell 2: 数据准备（纯 CPU，本地可直接运行）

> 分析字体覆盖率 → 渲染字形图片 → 提取训练/验证字符集。幂等，已完成会自动跳过。

In [ ]:
import os, json, subprocess

if not DO_DATA_PREP:
    print("DO_DATA_PREP=False，跳过 Cell 2")
else:
    def _load_state() -> dict:
        if os.path.exists(STATE_FILE):
            try:
                with open(STATE_FILE, "r", encoding="utf-8") as f:
                    return json.load(f)
            except Exception:
                return {}
        return {}

    def _save_state(s: dict) -> None:
        with open(STATE_FILE, "w", encoding="utf-8") as f:
            json.dump(s, f, ensure_ascii=False, indent=2)

    state = _load_state()
    data_done = os.path.isdir("data/reference") and os.path.isdir("data/target")
    splits_done = os.path.exists(f"charsets/splits/{FONT_NAME}/train.txt") and \
                  os.path.exists(f"charsets/splits/{FONT_NAME}/val.txt")
    same_font = state.get("data_font") == FONT_NAME

    if state.get("data_prep_done") and same_font and data_done and splits_done:
        print(f"字体 {FONT_NAME} 的数据准备已完成，跳过 Cell 2")
    else:
        print("\n===== 1. 分析字体覆盖率 =====")
        subprocess.run(["bash", "scripts/analyze_font.sh"], check=True)
        print("\n===== 2. 生成数据集图片 =====")
        subprocess.run(["bash", "scripts/prepare_dataset.sh"], check=True)
        print("\n===== 3. 提取训练/验证字符集 =====")
        subprocess.run(["bash", "scripts/extract_charset.sh"], check=True)

        if not (os.path.isdir("data/reference") and os.path.isdir("data/target")):
            raise RuntimeError("data/ 目录生成失败，请检查 prepare_dataset.sh")
        if not os.path.exists(f"charsets/splits/{FONT_NAME}/train.txt"):
            raise RuntimeError("train.txt 未生成，请检查 extract_charset.sh")

        state["data_prep_done"] = True
        state["data_font"] = FONT_NAME
        _save_state(state)
        print("\n===== 数据准备完成 =====")

---
## Cell 3: 训练 VQ-VAE（conservative 本地档，约 6-8 小时）

> 需要 NVIDIA GPU 或 Apple Silicon。断连/重启后重跑 Cell 0、Cell 1 即可从断点 epoch 精确续训。

In [ ]:
import os, subprocess

if not DO_TRAIN_VQVAE:
    print("DO_TRAIN_VQVAE=False，跳过 Cell 3")
elif not os.path.isdir("data"):
    print("data/ 不存在，请先运行 Cell 2 完成数据准备")
elif not check_training_viability()["viable"]:
    print("当前机器不满足训练条件（无 NVIDIA GPU / Apple Silicon），无法训练 VQ-VAE")
else:
    print(f"将执行训练脚本: {VQVAE_TRAIN_SCRIPT}（conservative 档，batch/workers 运行时自适应）")
    subprocess.run(["bash", VQVAE_TRAIN_SCRIPT], check=True)

---
## Cell 4-前置操作：离线放置 VGG16 权重（可选，联网下载慢时运行）

> **为什么需要**：Cell 4（训练 LDM 的 LPIPS 验证）与 Cell 5（LPIPS 指标）都会用到 VGG16 预训练权重（约 553MB），联网下载慢时可先手动下载放好。
>
> **把 `vgg16-397923af.pth` 直接丢进项目根目录即可**，本 Cell 会先直查这些固定位置（项目根 / `fonts/` / 上级目录 / 家目录 / 下载 / 桌面）——只做一次文件名拼接判断，**不遍历 C 盘 D 盘**；都没命中时才会再看一层子目录（可用 `SHALLOW_SEARCH=False` 关掉）。也可在 Cell 顶部直接填 `VGG16_SRC = "你的绝对路径"`，完全跳过搜索。
>
> 找到后移动到**本机** PyTorch Hub 缓存目录 `torch.hub.get_dir()/checkpoints`（Windows 通常为 `C:/Users/你/.cache/torch/hub/checkpoints`，macOS/Linux 为 `~/.cache/torch/hub/checkpoints`），后续直接命中、免下载。
>
> ⚠️ 这是 **VGG 预训练权重**，不是 `checkpoints/vqvae_{FONT_NAME}.pth`（本项目自己的模型），两者**不要混淆、不要放进 `checkpoints/`**。
>
> 📥 下载备份：`https://github.com/ICW-k/HanziGen_ICWfork/raw/main/vgg16-397923af.pth`
>
> 已联网且不在乎下载时间可跳过本 Cell，Cell 4 / Cell 5 会在首次使用时自动在线拉取。

In [ ]:
import os, shutil, torch

# ===== 离线放置 VGG16 权重：优先直查项目根目录，不遍历 C 盘 / D 盘 =====
VGG16_FILE = "vgg16-397923af.pth"

# ① 直接指定绝对路径（最省事，零搜索）：文件放哪儿就写哪儿
#    例：VGG16_SRC = "D:/weights/vgg16-397923af.pth"
VGG16_SRC = ""

# ② 直查目录（零遍历：只在这些目录里拼一次文件名判断存在性，瞬间完成）
DIRECT_DIRS = [
    os.getcwd(),                        # 项目根目录（最常见：直接丢在 HanziGen 根下）
    os.path.join(os.getcwd(), "fonts"), # 项目根/fonts
    os.path.dirname(os.getcwd()),       # 项目上级目录
    os.path.expanduser("~"),            # 家目录
    os.path.join(os.path.expanduser("~"), "Downloads"),   # 下载目录
    os.path.join(os.path.expanduser("~"), "Desktop"),     # 桌面
]

# ③ 上面都没命中时，是否允许再往下看一层子目录（仍不递归全盘）。不需要就设 False
SHALLOW_SEARCH = True

DST_DIR = os.path.join(torch.hub.get_dir(), "checkpoints")   # 跨平台：Windows / macOS / Linux 各自的用户缓存目录
os.makedirs(DST_DIR, exist_ok=True)
DST = os.path.join(DST_DIR, VGG16_FILE)

def locate(roots, depth: int):
    """depth=0: 只查 roots 本身；depth=1: 再看一层子目录。返回第一个命中的路径。"""
    for root in roots:
        if not os.path.isdir(root):
            continue
        if depth == 0:
            p = os.path.join(root, VGG16_FILE)
            if os.path.isfile(p):
                return p
            continue
        try:
            for entry in os.listdir(root):
                sub = os.path.join(root, entry)
                if not os.path.isdir(sub):
                    continue
                p = os.path.join(sub, VGG16_FILE)
                if os.path.isfile(p):
                    return p
        except (PermissionError, OSError):
            continue
    return None

if os.path.exists(DST):
    print(f"VGG16 权重已就位，跳过: {DST}")
else:
    src = VGG16_SRC if (VGG16_SRC and os.path.isfile(VGG16_SRC)) else None
    if src:
        print(f"[指定路径] 命中: {src}")
    else:
        src = locate(DIRECT_DIRS, depth=0)
        if src:
            print(f"[直查根目录] 命中: {src}")
        elif SHALLOW_SEARCH:
            src = locate(DIRECT_DIRS, depth=1)
            if src:
                print(f"[浅搜一层] 命中: {src}")

    if src is None:
        print(f"[WARN] 未找到 {VGG16_FILE}")
        print(f"  最省事：把文件放进项目根目录 {os.getcwd()}/ 后重跑本 Cell")
        print(f"  或在本 Cell 顶部填 VGG16_SRC = \"你的绝对路径/{VGG16_FILE}\"")
        print(f"  目标缓存位置: {DST}")
        print(f"  下载地址: https://github.com/ICW-k/HanziGen_ICWfork/raw/main/{VGG16_FILE}")
        print("  也可跳过本 Cell：Cell 4 / Cell 5 会在首次使用时自动联网下载（较慢）")
    else:
        if os.path.abspath(src) != os.path.abspath(DST):
            shutil.move(src, DST)
        print(f"VGG16 权重已就位: {DST}")
        print("  Cell 4 / Cell 5 的 LPIPS 将直接命中本地缓存，跳过远程下载")

---
## Cell 4: 训练 LDM（conservative 本地档，约 10-15 小时）

> 依赖 Cell 3 产物 `checkpoints/vqvae_{FONT_NAME}.pth`。

In [ ]:
import os, subprocess

vqvae_ckpt = f"checkpoints/vqvae_{FONT_NAME}.pth"

if not DO_TRAIN_LDM:
    print("DO_TRAIN_LDM=False，跳过 Cell 4")
elif not os.path.exists(vqvae_ckpt):
    print(f"{vqvae_ckpt} 不存在，请先完成 Cell 3 训练 VQ-VAE")
elif not check_training_viability()["viable"]:
    print("当前机器不满足训练条件（无 NVIDIA GPU / Apple Silicon），无法训练 LDM")
else:
    print("VQ-VAE 检查点确认：")
    print(f"  {vqvae_ckpt}")
    print(f"将执行训练脚本: {LDM_TRAIN_SCRIPT}（conservative 档，batch/workers 运行时自适应）")
    subprocess.run(["bash", LDM_TRAIN_SCRIPT], check=True)

---
## Cell 5: 推理生成 + 指标 + 转换 SVG

> 补字基准在 Cell 0 的 `CHARSET_BASE` 选择：`jf7000`（默认）/ `unihan` / `gbk`（简体 20,902 字）/ `gb2312`（核心 6,763 字）。
>
> 依赖 Cell 4 产物 `checkpoints/ldm_{FONT_NAME}.pth`。本 Cell 会自动把 `scripts/inference.sh` 的 `CHARSET_PATH` 指向所选基准的缺字表，并按本机硬件设置 `DEVICE`（cuda / mps / cpu）。

In [ ]:
import os, re, subprocess
from utils.hardware import detect_hardware

ldm_ckpt = f"checkpoints/ldm_{FONT_NAME}.pth"

if not DO_INFERENCE:
    print("DO_INFERENCE=False，跳过 Cell 5")
elif not os.path.exists(ldm_ckpt):
    print(f"{ldm_ckpt} 不存在，请先完成 Cell 4 训练 LDM")
else:
    # ===== 1. 按补字基准确定字符集文件 =====
    def build_charset_path(base: str) -> str:
        if base in ("gbk", "gb2312"):
            # 用系统编码器直接计算 GBK / GB2312 缺失字，无需额外字集文件
            from fontTools.ttLib import TTFont
            base_chars = set()
            for cp in list(range(0x3400, 0x4DC0)) + list(range(0x4E00, 0xA000)):
                try:
                    chr(cp).encode(base)
                    base_chars.add(chr(cp))
                except UnicodeEncodeError:
                    pass
            font = TTFont(f"fonts/{TARGET_FONT}", fontNumber=0)
            cmap = set()
            for table in font["cmap"].tables:
                if table.isUnicode():
                    cmap.update(table.cmap.keys())
            missing = sorted(base_chars - {chr(c) for c in cmap})
            out_dir = f"charsets/{base}_coverage/{FONT_NAME}"
            os.makedirs(out_dir, exist_ok=True)
            out_path = f"{out_dir}/missing.txt"
            with open(out_path, "w", encoding="utf-8") as f:
                f.write("\n".join(missing))
            print(f"[{base}] 基准 {len(base_chars)} 字，字体缺失 {len(missing)} 字 → {out_path}")
            return out_path
        if base == "unihan":
            p = f"charsets/unihan_coverage/{FONT_NAME}/missing.txt"
        else:  # jf7000
            p = f"charsets/jf7000_coverage/{FONT_NAME}/missing.txt"
        if not os.path.exists(p):
            raise FileNotFoundError(f"{p} 不存在，请先运行 Cell 2 完成字体分析")
        return p

    charset_path = build_charset_path(CHARSET_BASE)
    print(f"补字基准: {CHARSET_BASE} → {charset_path}")

    # ===== 2. inference.sh 指向该字符集，并按本机硬件设置设备 =====
    info = detect_hardware()
    device = "cuda" if info["gpu_available"] else ("mps" if info["mps_available"] else "cpu")
    with open("scripts/inference.sh", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r'CHARSET_PATH="[^"]*"', f'CHARSET_PATH="{charset_path}"', content)
    content = re.sub(r'DEVICE="[^"]*"', f'DEVICE="{device}"', content)
    with open("scripts/inference.sh", "w", encoding="utf-8") as f:
        f.write(content)
    print(f"inference.sh 已指向补字基准字符集（DEVICE={device}）")

    # ===== 3. 推理生成（关键步骤） =====
    print("\n===== 推理生成 =====")
    subprocess.run(["bash", "scripts/inference.sh"], check=True)

    # ===== 补字数量自检 =====
    gen_dir = f"samples_{FONT_NAME}/inference/gen"
    if os.path.isdir(gen_dir):
        gen_pngs = [p for p in os.listdir(gen_dir) if p.endswith(".png")]
        print(f"\n===== 推理结果自检 =====")
        print(f"实际生成 PNG: {len(gen_pngs)} 张（位于 {gen_dir}/）")
    else:
        print(f"[WARN] 未找到生成目录 {gen_dir}")

    # 计算评估指标（非关键，失败不阻断）
    print("\n===== 计算评估指标 =====")
    with open("scripts/compute_metrics.sh", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r'DEVICE="[^"]*"', f'DEVICE="{device}"', content)
    with open("scripts/compute_metrics.sh", "w", encoding="utf-8") as f:
        f.write(content)
    try:
        subprocess.run(["bash", "scripts/compute_metrics.sh"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"[WARN] 评估指标计算失败（返回码 {e.returncode}），不影响补字结果。")

    # 转换 SVG
    print("\n===== 转换 SVG =====")
    subprocess.run(["bash", "scripts/convert_to_svg.sh"], check=True)
    print(f"\n===== 全部完成！SVG 输出在 svgs_{FONT_NAME}/ 目录 =====")

---
## 常见问题

| 问题 | 处理 |
|---|---|
| Cell 1 提示「无法训练」 | 你的机器无 NVIDIA GPU / Apple Silicon（如仅 AMD 核显）。训练需改用云端 GPU 实例，数据准备与 SVG 转换仍可本地跑 |
| 训练 OOM | 优先在 Cell 0 调小 `VRAM_RESERVE_FRACTION`（如 0.70）后重跑 Cell 1 + Cell 3；也可手动调低 `scripts/train_vqvae_local.sh` 的 `BATCH_SIZE`（把 `auto` 改为具体整数） |
| 想少留冗余提速 | Cell 0 设 `VRAM_RESERVE_FRACTION = 0.92`，重跑 Cell 1 + Cell 3（仅影响 VQ-VAE batch，LDM 不受影响） |
| 想压榨性能 | 把 local 脚本的 `PRESET=conservative` 改为 `PRESET=aggressive`（会占用更多 CPU，适合专用训练机） |
| 只想补 GBK / GB2312 简体缺字 | Cell 0 设 `CHARSET_BASE="gbk"` 或 `"gb2312"`（训练完成后只跑 Cell 5 即可，会自动生成缺字表并改写 `inference.sh`） |
| 换基准后补字数量没变 | Cell 5 每次运行都会按 `CHARSET_BASE` 重新定位字表并改写 `scripts/inference.sh` 的 `CHARSET_PATH`，重跑 Cell 5 即可 |
| unihan / gbk 补字太慢 | `scripts/inference.sh` 的 `SAMPLE_STEPS=50` 改为 `20`（速度约 2.5 倍，质量略降） |
| 换字体后想重训 | 删除 `checkpoints/vqvae_{FONT_NAME}.pth`、`ldm_{FONT_NAME}.pth` 与 `colab_state.json`，重跑 Cell 1 |